In [4]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import librosa
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="simon3000/genshin-voice", 
                  repo_type="dataset", local_dir="./genshin-voice")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 53 files:  21%|██        | 11/53 [01:40<05:21,  7.65s/it]

  2025-09-26T08:31:33.366314Z  WARN  Status Code: 503. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-26T08:31:33.366367Z  WARN  Retry attempt #0. Sleeping 1.648815107s before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171

  2025-09-26T08:31:35.018780Z  WARN  Status Code: 503. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-26T08:31:35.018818Z  WARN  Retry attempt #1. Sleeping 3.221156095s before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171

  2025-09-26T08:31:38.246625Z  WARN  Status Code: 503. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-26T08:31:38.246654Z  WARN  Retry attempt #2. Sleeping 4.691001028s before the next atte

Fetching 53 files:  26%|██▋       | 14/53 [02:25<06:47, 10.45s/it]

  2025-09-26T08:31:42.947586Z  WARN  Status Code: 503. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-26T08:31:42.947614Z  WARN  Retry attempt #3. Sleeping 21.160873977s before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171



Fetching 53 files:  40%|███▉      | 21/53 [03:04<03:26,  6.46s/it]

  2025-09-26T08:32:45.552663Z  WARN  Reqwest(reqwest::Error { kind: Request, url: "https://transfer.xethub.hf.co/xorbs/default/42f2b1d86ba3e732c9a5aa7e6bc5fd959418d661b40b4d495b174e59d08562ec?X-Xet-Signed-Range=bytes%3D29931708-29972410&X-Xet-Session-Id=01K62JD6TWD1MZ3FTV2CTNQ5X8&Expires=1758879124&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly90cmFuc2Zlci54ZXRodWIuaGYuY28veG9yYnMvZGVmYXVsdC80MmYyYjFkODZiYTNlNzMyYzlhNWFhN2U2YmM1ZmQ5NTk0MThkNjYxYjQwYjRkNDk1YjE3NGU1OWQwODU2MmVjP1gtWGV0LVNpZ25lZC1SYW5nZT1ieXRlcyUzRDI5OTMxNzA4LTI5OTcyNDEwJlgtWGV0LVNlc3Npb24tSWQ9MDFLNjJKRDZUV0QxTVozRlRWMkNUTlE1WDgiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NTg4NzkxMjR9fX1dfQ__&Signature=WTWwaoKomW3E0rbSPruDQrx4ayFzKVG-nmjQHnYLwQhEieDjclpHlFRUexHD2CzkR7cn8cSMF9KlWJfcUI5OlEuL7JWipLiH84DLkfa1Pj3x6Q49ThKmKtr8Tz-XjTpX98Qh7kIpzfIG-Er9rMjKT3Q8jJFWjiwCbbl6RORtcGAnX1EfaHYealKMqpqWdPMdVviLSH-nciM~H979S8Acxf5rB1tBZvKTB7~ounpmjdi9ocZCzEKMTxMSD4V9HU1EJX~eAru83yMgiLY46fWhnU-BOZLUO3QeBFuaH~RwGE

Fetching 53 files: 100%|██████████| 53/53 [08:49<00:00,  9.98s/it]


'/home/ubuntu/genshin-voice'

In [3]:
files = glob('genshin-voice/*/*.parquet')
len(files)

50

In [8]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in files:
        base = f.split('/')[0] + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in tqdm(range(len(df))):
            t = df['transcription'].iloc[i].strip()
            if len(t) < 2:
                continue
            speaker = df['speaker'].iloc[i].strip()
            if len(speaker) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = librosa.load(io.BytesIO(b), sr = 24000)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{speaker}"
            })
        
    return data

In [7]:
data = loop((files[:1], 0))
data

  0%|          | 0/8480 [00:12<?, ?it/s]


[{'audio_filename': 'genshin-voice_audio/genshin-voice-data-train-00018-of-00050_0.mp3',
  'text': '得了得了，每一个刚来到这里找我登记的犯人，都是愁眉苦脸的，我哪里还笑得出来啊…',
  'speaker': 'genshin-voice_audio_Marette'}]

In [9]:
data = multiprocessing(files, loop, cores = 10)

100%|██████████| 8480/8480 [07:58<00:00, 17.74it/s]


In [10]:
len(data)

373435

In [11]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'genshin-voice_audio/genshin-voice-data-train-00018-of-00050_0.mp3',
 'text': '得了得了，每一个刚来到这里找我登记的犯人，都是愁眉苦脸的，我哪里还笑得出来啊…',
 'speaker': 'genshin-voice_audio_Marette'}

In [12]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'genshin-voice')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  6.57ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  46%|████▌     | 14.7MB / 32.0MB, 1.47MB/s  
Processing Files (0 / 1):  99%|█████████▉| 31.7MB / 32.0MB, 3.10MB/s  
Processing Files (1 / 1): 100%|██████████| 32.0MB / 32.0MB, 3.14MB/s  
Processing Files (1 / 1): 100%|██████████| 32.0MB / 32.0MB, 3.20MB/s  
New Data Upload: 100%|██████████| 32.0MB / 32.0MB, 3.20MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:11<00:00, 11.65s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/0392ec841e023e1c45986b4f7fcb3ea0db7e6e13', commit_message='Upload dataset', commit_description='', oid='0392ec841e023e1c45986b4f7fcb3ea0db7e6e13', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [13]:
audio_files = [d['audio_filename'] for d in data]

with open('genshin-voice-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [16]:
!zip -rq genshin-voice_audio.zip genshin-voice_audio

In [18]:
# !hf upload malaysia-ai/Multilingual-TTS genshin-voice_audio.zip --repo-type=dataset

In [21]:
# !zip -rq genshin-voice_audio_neucodec.zip genshin-voice_audio_neucodec

In [22]:
# !hf upload malaysia-ai/Multilingual-TTS genshin-voice_audio_neucodec.zip --repo-type=dataset